# Ligação média

## Deslocamento vertical, dV

### Temperatura

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Grelha: {grid_size} m x {grid_size} m',
#     f'Clustering: DTW + Hierárquico (ligação média)',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)

ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais médias por cluster + temperatura ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais do cluster (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário, preta)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais se ainda não existir
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))
clusters_present = sorted(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """Determina a cor de cada ramo do dendrograma conforme os clusters."""
    if max(y_coords) > cut_distance:
        return "#A0A0A0"  # acima do corte
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]
    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    return "#A0A0A0"

# ---------------------------------------------------
# FIGURA 1: Dendrograma + Mapa com legenda e descrição técnica
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.02)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)
leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round', zorder=3)

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW', zorder=2)
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")

# Colorir labels das folhas conforme o cluster
ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors.get(cluster_id, "#333333"))

ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# 👉 Adiciona esta linha para incluir na legenda
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id} - {len(cluster_df[cluster_df["cluster"]==cluster_id])} células')

# # --- Caixa técnica
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico (método "average")',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)

ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# FIGURA 2: Séries médias por cluster + temperatura
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(clusters_present):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cluster_cells) == 0:
        ax.set_title(f'Cluster {cluster_id} - 0 células')
        continue

    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.8,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()



### Nível da albufeira

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols,
        var_name='date',
        value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Nível da albufeira
# ==============================
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel['nivel_smooth'] = df_nivel['nivel']  # sem suavização

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

condensed = squareform(dist_matrix)
Z = linkage(condensed, method='average')

distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Título e legenda
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais (dV + nível) ---
global_dV_min = agg_pivot.min().min()
global_dV_max = agg_pivot.max().max()
dV_margin = (global_dV_max - global_dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais em cinzento
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid],
                color='lightgray', alpha=0.6)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color,
            linewidth=2.5, label=f'Média Cluster {cluster_id}')

    # Nível da albufeira (eixo secundário)
    ax2 = ax.twinx()
    nivel_min, nivel_max = df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (global_dV_max - global_dV_min) * 0.2 + (global_dV_max - (global_dV_max - global_dV_min) * 0.25)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2, alpha=0.8,
             label='Nível da albufeira (m)')

    # Eixo do nível
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')
    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (global_dV_max - global_dV_min) * 0.2 + (global_dV_max - (global_dV_max - global_dV_min) * 0.25)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{v:.0f}" for v in nivel_ticks_real])

    # Limites verticais coerentes
    ax.set_ylim(global_dV_min - dV_margin, global_dV_max + dV_margin)
    ax2.set_ylim(global_dV_min - dV_margin, global_dV_max + dV_margin)

    # Estilo e legendas
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais (se ainda não tiveres)
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """
    Determina a cor do ramo com base nos clusters das leaves que ele conecta.
    """
    if max(y_coords) > cut_distance:
        return "#A0A0A0"

    # Aproxima leaves conectadas
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]

    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    else:
        return "#A0A0A0"

# ---------------------------------------------------
# Figura 1: Dendrograma + Mapa
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.01)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)

leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round')

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")
ax_dendro.legend(fontsize=10)

ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors[cluster_id])
    
ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# Figura 2: Séries médias por cluster com nível
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(sorted(cluster_df['cluster'].unique())):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Nível da albufeira (escala visual)
    ax2 = ax.twinx()
    nivel_min, nivel_max = df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.2 + (dV_max - (dV_max - dV_min) * 0.25)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2, alpha=0.8, zorder=10,
             label='Nível da albufeira (m)')
    
    ax2.set_ylabel("Nível da albufeira (m)", fontsize=11)

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)

    # Ticks reais para o eixo do nível
    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.2 + (dV_max - (dV_max - dV_min) * 0.25)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{v:.1f}" for v in nivel_ticks_real])

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### Precipitação total

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")  # teu arquivo de precipitação
df_prec['data'] = pd.to_datetime(df_prec['data'])
# Não precisa de suavização, usaremos valores brutos


# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Título e legenda
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# -----------------------------
# Séries temporais médias com precipitação
# -----------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Precipitação total (barra) como eixo secundário
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    # Limites e formatação eixo principal
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')


plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais se ainda não existir
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))
clusters_present = sorted(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """Determina a cor de cada ramo do dendrograma conforme os clusters."""
    if max(y_coords) > cut_distance:
        return "#A0A0A0"  # acima do corte
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]
    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    return "#A0A0A0"

# ---------------------------------------------------
# FIGURA 1: dendrograma + mapa
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.01)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)
leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round', zorder=3)

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW', zorder=2)
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")
ax_dendro.legend(fontsize=10)

# Colorir labels das folhas
ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors[cluster_id])

ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total.",
    fontsize=16
)
ax_map.set_axis_off()

plt.tight_layout()
plt.show()


# ---------------------------------------------------
# FIGURA 2: Séries médias por cluster + precipitação
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(clusters_present):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cluster_cells) == 0:
        ax.set_title(f'Cluster {cluster_id} - 0 células')
        continue

    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.8,
            label=f'Média Cluster {cluster_id}')

    # Precipitação (eixo secundário)
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=20, color='blue', alpha=0.3, label='Precipitação (mm)')
    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2.set_ylim(0, df_prec['prec'].max() * 1.1)

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Eixos e título
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()



### Precipitação total acumulada

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")  # teu arquivo de precipitação
df_prec['data'] = pd.to_datetime(df_prec['data'])
# Não precisa de suavização, usaremos valores brutos


# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Título e legenda
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# -----------------------------
# Séries temporais médias com precipitação acumulada e mensal
# -----------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Ordenar e preparar precipitação
df_prec = df_prec.sort_values('data')
df_prec['prec_acum'] = df_prec['prec'].cumsum()

# Precipitação mensal
df_prec_monthly = df_prec.set_index('data').resample('M')['prec'].sum().reset_index()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais de dV
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Eixo secundário para precipitação
    ax2 = ax.twinx()
    # Linha acumulada
    ax2.plot(df_prec['data'], df_prec['prec_acum'], color='blue', linewidth=2, alpha=0.7,
             label='Precipitação acumulada (mm)')
    # Barras mensais
    ax2.bar(df_prec_monthly['data'], df_prec_monthly['prec'], width=20, color='lightblue', alpha=0.5,
            label='Precipitação mensal (mm)')

    ax2.set_ylabel("Precipitação (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax2.set_ylim(0, max(df_prec['prec_acum'].max(), df_prec_monthly['prec'].max()*1.2))

    # Limites e formatação eixo principal
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais (se ainda não tiveres)
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """
    Determina a cor do ramo com base nos clusters das leaves que ele conecta.
    """
    if max(y_coords) > cut_distance:
        return "#A0A0A0"

    # Aproxima leaves conectadas
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]

    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    else:
        return "#A0A0A0"

# ---------------------------------------------------
# Figura 1: Dendrograma + Mapa
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.01)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)

leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round')

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")
ax_dendro.legend(fontsize=10)

ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors[cluster_id])

ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# Figura 2: Séries médias por cluster com precipitação acumulada
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

# Calcular precipitação acumulada
df_prec = df_prec.sort_values('data')
df_prec['prec_acum'] = df_prec['prec'].cumsum()

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(sorted(cluster_df['cluster'].unique())):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Precipitação acumulada (linha) no eixo secundário
    ax2 = ax.twinx()
    ax2.plot(df_prec['data'], df_prec['prec_acum'], color='blue', linewidth=2, alpha=0.8,
             label='Precipitação acumulada (mm)')
    ax2.set_ylabel("Precipitação acumulada (mm)", color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')

    # Escala real da precipitação acumulada
    ax2.set_ylim(0, df_prec['prec_acum'].max() * 1.1)

    # Limites e formatação eixo principal
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)

    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### Precipitação total anual acumulada

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
import matplotlib.dates as mdates
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Pontos centrais
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# Agrupar dV/dH
agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar precipitação
# ==============================
df_prec = pd.read_excel("data/prec.xlsx")
df_prec['data'] = pd.to_datetime(df_prec['data'])
df_prec['ano_hidrologico'] = df_prec['data'].apply(lambda x: x.year+1 if x.month>=10 else x.year)
df_prec['prec_acumulada'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Matriz de distâncias DTW
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])
for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

condensed = squareform(dist_matrix)
Z = linkage(condensed, method='average')
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte DTW+Hierárquico em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}
grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Título e legenda
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total anual acumulada.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# Séries temporais médias + precipitação
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    ax2 = ax.twinx()
    # Barras mensais
    ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='deepskyblue', alpha=0.5, label='Precipitação mensal')
    # Linha acumulada anual
    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'], color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.set_ylabel('dV (mm)')
    ax2.set_ylabel('Precipitação (mm)', color='blue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    by_label = dict(zip(labels1 + labels2, lines1 + lines2))
    ax.legend(by_label.values(), by_label.keys(), fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais (se ainda não tiveres)
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    if max(y_coords) > cut_distance:
        return "#A0A0A0"

    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]

    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    else:
        return "#A0A0A0"

# ---------------------------------------------------
# Figura 1: Dendrograma + Mapa
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.01)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)

leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round')

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")
ax_dendro.legend(fontsize=10)

ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors[cluster_id])

ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a precipitação total acumulada.",
    fontsize=16
)
ax_map.set_axis_off()

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# Figura 2: Séries médias por cluster + precipitação
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(sorted(cluster_df['cluster'].unique())):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Precipitação mensal (barras) + acumulada anual (linha)
    ax2 = ax.twinx()
    ax2.bar(df_prec['data'], df_prec['prec'], width=15, color='deepskyblue', alpha=0.5, label='Precipitação mensal')

    for ano, grupo in df_prec.groupby('ano_hidrologico'):
        ax2.plot(grupo['data'], grupo['prec_acumulada'], color='blue', linewidth=2, alpha=0.9,
                 label=f'Acumulado {ano}')

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(0, df_prec['prec_acumulada'].max() * 1.1)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    by_label = dict(zip(labels1 + labels2, lines1 + lines2))
    ax.legend(by_label.values(), by_label.keys(), fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


# Ligação completa

## Deslocamento vertical, dV

### Temperatura

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='complete')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Grelha: {grid_size} m x {grid_size} m',
#     f'Clustering: DTW + Hierárquico (ligação média)',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação completa.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais médias por cluster + temperatura ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais do cluster (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário, preta)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais se ainda não existir
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))
clusters_present = sorted(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """Determina a cor de cada ramo do dendrograma conforme os clusters."""
    if max(y_coords) > cut_distance:
        return "#A0A0A0"  # acima do corte
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]
    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    return "#A0A0A0"

# ---------------------------------------------------
# FIGURA 1: Dendrograma + Mapa com legenda e descrição técnica
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.02)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)
leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round', zorder=3)

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW', zorder=2)
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")

# Colorir labels das folhas conforme o cluster
ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors.get(cluster_id, "#333333"))

ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# --- Pontos centrais
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# 👉 Adiciona esta linha para incluir na legenda
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

# --- Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# --- Legenda dos clusters
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id} - {len(cluster_df[cluster_df["cluster"]==cluster_id])} células')

# # --- Caixa técnica
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico (método "average")',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação completa.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# FIGURA 2: Séries médias por cluster + temperatura
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1
temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(clusters_present):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    if len(cluster_cells) == 0:
        ax.set_title(f'Cluster {cluster_id} - 0 células')
        continue

    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.8,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.0f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### Nível da albufeira

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols,
        var_name='date',
        value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Nível da albufeira
# ==============================
df_nivel = pd.read_excel("data/alqueva_nivel.xlsx")
df_nivel['data'] = pd.to_datetime(df_nivel['data'])
df_nivel['nivel_smooth'] = df_nivel['nivel']  # sem suavização

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
agg_pivot = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot.index.tolist()
X = agg_pivot.values.astype(float)
n = X.shape[0]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

condensed = squareform(dist_matrix)
Z = linkage(condensed, method='complete')

distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Tamanho da grelha: {grid_size} x {grid_size} m',
#     f'Técnica de clustering: DTW + Hierárquico',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# Título e legenda
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação completa.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais (dV + nível) ---
global_dV_min = agg_pivot.min().min()
global_dV_max = agg_pivot.max().max()
dV_margin = (global_dV_max - global_dV_min) * 0.1

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais em cinzento
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid],
                color='lightgray', alpha=0.6)

    # Média do cluster (linha colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color,
            linewidth=2.5, label=f'Média Cluster {cluster_id}')

    # Nível da albufeira (eixo secundário)
    ax2 = ax.twinx()
    nivel_min, nivel_max = df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (global_dV_max - global_dV_min) * 0.2 + (global_dV_max - (global_dV_max - global_dV_min) * 0.25)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2, alpha=0.8,
             label='Nível da albufeira (m)')

    # Eixo do nível
    ax2.set_ylabel("Nível da albufeira (m)", color='black')
    ax2.tick_params(axis='y', labelcolor='black')
    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (global_dV_max - global_dV_min) * 0.2 + (global_dV_max - (global_dV_max - global_dV_min) * 0.25)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{v:.0f}" for v in nivel_ticks_real])

    # Limites verticais coerentes
    ax.set_ylim(global_dV_min - dV_margin, global_dV_max + dV_margin)
    ax2.set_ylim(global_dV_min - dV_margin, global_dV_max + dV_margin)

    # Estilo e legendas
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.set_xlabel('Ano')
    ax.set_ylabel('dV (mm)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, fcluster
import geopandas as gpd
import contextily as ctx
import numpy as np
import pandas as pd

# ---------------------------------------------------
# Determinar clusters finais (se ainda não tiveres)
# ---------------------------------------------------
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
n_clusters = len(np.unique(cluster_labels))

# ---------------------------------------------------
# Função auxiliar: cor do ramo com base nos clusters ligados
# ---------------------------------------------------
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """
    Determina a cor do ramo com base nos clusters das leaves que ele conecta.
    """
    if max(y_coords) > cut_distance:
        return "#A0A0A0"

    # Aproxima leaves conectadas
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]

    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    else:
        return "#A0A0A0"

# ---------------------------------------------------
# Figura 1: Dendrograma + Mapa
# ---------------------------------------------------
fig1 = plt.figure(figsize=(24, 12))
gs1 = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.01)

# --- Dendrograma (esquerda)
ax_dendro = fig1.add_subplot(gs1[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)

leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.5, solid_capstyle='round')

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")
ax_dendro.legend(fontsize=10)

ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors[cluster_id])
    
ax_dendro.set_title("Dendrograma", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig1.add_subplot(gs1[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação completa.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e o nível da albufeira.",
    fontsize=16
)
ax_map.set_axis_off()

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# Figura 2: Séries médias por cluster com nível
# ---------------------------------------------------
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

fig2, axes = plt.subplots(1, n_clusters, figsize=(24, 8), sharey=True)

for idx, cluster_id in enumerate(sorted(cluster_df['cluster'].unique())):
    ax = axes[idx]
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot.loc[cluster_cells]

    # Séries individuais (cinzento)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.6)

    # Média colorida
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Nível da albufeira (escala visual)
    ax2 = ax.twinx()
    nivel_min, nivel_max = df_nivel['nivel_smooth'].min(), df_nivel['nivel_smooth'].max()
    nivel_visual = (df_nivel['nivel_smooth'] - nivel_min) / (nivel_max - nivel_min)
    nivel_visual = nivel_visual * (dV_max - dV_min) * 0.2 + (dV_max - (dV_max - dV_min) * 0.25)
    ax2.plot(df_nivel['data'], nivel_visual, color='black', linewidth=2, alpha=0.8, zorder=10,
             label='Nível da albufeira (m)')
    
    ax2.set_ylabel("Nível da albufeira (m)", fontsize=11)

    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)

    # Ticks reais para o eixo do nível
    nivel_ticks_real = np.linspace(nivel_min, nivel_max, 6)
    nivel_ticks_visual = (nivel_ticks_real - nivel_min) / (nivel_max - nivel_min)
    nivel_ticks_visual = nivel_ticks_visual * (dV_max - dV_min) * 0.2 + (dV_max - (dV_max - dV_min) * 0.25)
    ax2.set_yticks(nivel_ticks_visual)
    ax2.set_yticklabels([f"{v:.1f}" for v in nivel_ticks_real])

    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


### Precipitação total